In [9]:
%pip install "gspread<6" --force-reinstall
%pip install --force-reinstall eep153_tools gspread_pandas python_gnupg

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 43.6 MB/s eta 0:00:00
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.3.0
    Uninstalling urllib3-2.3.0:
      Successfully uninstalled urllib3-2.3.0
  Attempting uninstall: pycparser
    Found existing installation: pycparser 2.22
    Uninstalling pycparser-2.22:
      Successfully uninstalled pycparser-2.22━━━━━━━━━━━━━━━━━━━━━  1/15 [pycparser]
  Attempting uninstall: pyasn1━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/15 [pycparser]
    Found existing installation: pyasn1 0.6.2━━━━━━━━━━━━━━━━━  1/15 [pycparser]
    Uninstalling pyasn1-0.6.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/15 [pycparser]
      Successfully uninstalled pyasn1-0.6.2━━━━━━━━━━━━━━━━━━━  1/15 [pycparser]
  Attempting uninstall: oauthlib━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/15 [pyasn1]
    Found existing installation: oauthlib

In [13]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog
from eep153_tools.sheets import read_sheets

In [ ]:
data_url = "https://docs.google.com/spreadsheets/d/1m34r0of56QSNEXVFS7MqcvwsX_EUrTNBYlZAuuQFAto/edit?usp=sharing"

In [15]:
recipe_totals_raw = read_sheets(data_url, sheet="fresh pre-transpose")

In [16]:
# loading prices
prices_raw = read_sheets(data_url, sheet="prices")
prices = prices_raw.copy()
prices["Name"] = prices["Name"].astype(str).str.strip().str.lower()
prices = prices.set_index("Name")[["price_per_serv"]]

In [17]:
# matching foods 
recipe_totals_raw.columns = recipe_totals_raw.columns.astype(str).str.strip().str.lower()
common_foods = recipe_totals_raw.columns.intersection(prices.index)

In [18]:
A_all = recipe_totals_raw[common_foods].copy()
prices = prices.loc[common_foods].copy()
c = prices["price_per_serv"].astype(float)

In [19]:
#nutrient names as indexes
A_all.index = recipe_totals_raw.iloc[:, 0]
A_all = A_all.iloc[:, 1:]
A_all.index = A_all.index.astype(str).str.strip()

In [33]:
# cleaning names
def clean_nutrient_name(x):
    x = str(x)
    x = x.replace('"', '')
    x = x.replace("'", '')
    x = x.strip()
    x = x.rstrip(',')
    x = " ".join(x.split())
    return x

A_all.index = A_all.index.map(clean_nutrient_name)

In [32]:
rename_map = {
    "Fiber, total dietary": "Dietary Fiber",
    "Calcium, Ca": "Calcium",
    "Iron, Fe": "Iron",
    "Magnesium, Mg": "Magnesium",
    "Phosphorus, P": "Phosphorus",
    "Potassium, K": "Potassium",
    "Sodium, Na": "Sodium",
    "Zinc, Zn": "Zinc",
    "Copper, Cu": "Copper",
    "Selenium, Se": "Selenium",
    "Vitamin A, RAE": "Vitamin A",
    "Vitamin E (alpha-tocopherol)": "Vitamin E",
    "Vitamin D (D2 + D3)": "Vitamin D",
    "Vitamin C, total ascorbic acid": "Vitamin C",
    "Vitamin B-6": "Vitamin B6",
    "Vitamin B-12": "Vitamin B12",
    "Folate, DFE": "Folate",
    "Choline, total": "Choline",
    "18:2 n-6 c,c": "Linoleic Acid",
    "18:3 n-3 c,c,c (ALA)": "Linolenic Acid",
    "Energy": "Energy",
    "Protein": "Protein",
    "Carbohydrate, by difference": "Carbohydrate"}

A_all = A_all.rename(index=rename_map)

In [31]:
# loading RDA
rda = read_sheets(data_url, sheet="rda")
rda = rda.set_index("Nutrient")
target_rda = rda["Male_19_30"].dropna()
target_rda.index = target_rda.index.map(clean_nutrient_name)

In [26]:
# matching nutrients
common_nutrients = A_all.index.intersection(target_rda.index)

A_check = A_all.loc[common_nutrients].copy()
A_check = A_check.apply(pd.to_numeric, errors="coerce").fillna(0)

b_check = pd.to_numeric(target_rda.loc[common_nutrients], errors="coerce")

In [27]:
# solving diet with ingredients
A_min = -A_check.values
b_min = -b_check.values

result = linprog(c.values, A_ub=A_min, b_ub=b_min,bounds=(0, None), method="highs")

print("Success:", result.success)
print("Status:", result.message)

Success: True
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)


In [28]:
# solution 
solution = pd.Series(result.x, index=c.index)
solution = solution[solution > 1e-6]

print("\nOptimal ingredient amounts:")
print(solution)

total_cost = (solution * c.loc[solution.index]).sum()
print("\nTotal daily cost:", total_cost)


Optimal ingredient amounts:
cream           9.019970
garlic         16.229344
fenugreek       4.733128
feta cheese     1.420118
olive oil       0.246299
dtype: float64

Total daily cost: 3.696977622461867


In [29]:
nutrient_totals = A_check @ result.x

nutrient_df = pd.DataFrame({"obtained": nutrient_totals,"required": b_check}, index=A_check.index)

nutrient_df["met?"] = nutrient_df["obtained"] >= nutrient_df["required"]

print("\nNutrient check:")
print(nutrient_df)


Nutrient check:
                   obtained  required   met?
Dietary Fiber    160.254179      33.6   True
Calcium         5021.124669    1000.0   True
Energy         13607.575698    2400.0   True
Carbohydrate     773.631474     130.0   True
Vitamin A       3141.244502     900.0   True
Vitamin C        181.904808      90.0   True
Iron             187.352673       8.0   True
Sodium          2400.445201    2300.0   True
Protein          254.700425      56.0   True
Potassium      11115.706377    4700.0   True
Magnesium       1390.863133     400.0   True
Phosphorus      4876.813756     700.0   True
Zinc              36.733194      11.0   True
Copper            10.151790       0.9   True
Selenium         237.227966      55.0   True
Vitamin E         15.000000      15.0  False
Vitamin D         15.000000      15.0  False
Thiamin            5.277273       1.2   True
Riboflavin         6.411887       1.3   True
Niacin            21.080426      16.0   True
Vitamin B6        23.846013       1.3 

In [30]:
final_diet = pd.DataFrame({ "food": solution.index, "servings": solution.values,"price_per_serv": c.loc[solution.index].values})

final_diet["daily_cost"] = final_diet["servings"] * final_diet["price_per_serv"]

print("\nFinal diet table:")
print(final_diet)


Final diet table:
          food   servings  price_per_serv  daily_cost
0        cream   9.019970          0.0925    0.834347
1       garlic  16.229344          0.0828    1.343790
2    fenugreek   4.733128          0.0874    0.413675
3  feta cheese   1.420118          0.7367    1.046201
4    olive oil   0.246299          0.2394    0.058964
